# Lesson 5.2 — Helm Test

**Phase 5: Real-World Workflows**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Write Helm test hooks that verify a deployed release works correctly
- Run `helm test` against a live release
- Understand the `test` hook lifecycle
- Design smoke tests and connectivity tests for your charts

---

## What Is `helm test`?

`helm test` runs templates annotated with `"helm.sh/hook": test` against a deployed release. These are Pod or Job resources that run and exit 0 on success.

Test hooks are the closest thing Helm has to **integration tests** — they run against the live deployment and verify the app is working.

> **Reference:** [Chart Tests](https://helm.sh/docs/topics/chart_tests/) — https://helm.sh/docs/topics/chart_tests/

---

## The Test Hook Lifecycle

```
helm test <release-name>
    |
    ├─ Helm finds all templates annotated with 'helm.sh/hook: test'
    ├─ Deploys them as Pods/Jobs
    ├─ Waits for them to complete
    │     exit 0 = PASS
    │     exit != 0 = FAIL
    └─ Reports results
```

---

## Convention: Place Tests in `templates/tests/`

By convention (not requirement), test files go in `templates/tests/`. This keeps them visually separate from the main resources.

---

## Setup: Deploy Something to Test

In [ ]:
rm -rf /tmp/testchart && mkdir -p /tmp/testchart/templates/tests

cat > /tmp/testchart/Chart.yaml << 'EOF'
apiVersion: v2
name: testchart
version: 0.1.0
appVersion: "1.0"
EOF

cat > /tmp/testchart/values.yaml << 'EOF'
service:
  name: ""
  port: 80
image:
  repository: nginx
  tag: "1.25"
EOF

# A simple nginx Deployment
cat > /tmp/testchart/templates/deployment.yaml << 'EOF'
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {{ .Release.Name }}-nginx
spec:
  replicas: 1
  selector:
    matchLabels:
      app: {{ .Release.Name }}-nginx
  template:
    metadata:
      labels:
        app: {{ .Release.Name }}-nginx
    spec:
      containers:
        - name: nginx
          image: {{ .Values.image.repository }}:{{ .Values.image.tag }}
          ports:
            - containerPort: 80
EOF

# A Service to expose it
cat > /tmp/testchart/templates/service.yaml << 'EOF'
apiVersion: v1
kind: Service
metadata:
  name: {{ .Release.Name }}-nginx
spec:
  selector:
    app: {{ .Release.Name }}-nginx
  ports:
    - port: {{ .Values.service.port }}
      targetPort: 80
EOF

echo "Chart ready"

---

## Writing Test Hooks

### Test 1: Connectivity Test

Check that the Service is reachable from inside the cluster:

In [ ]:
cat > /tmp/testchart/templates/tests/test-connection.yaml << 'EOF'
apiVersion: v1
kind: Pod
metadata:
  name: {{ .Release.Name }}-test-connection
  annotations:
    # This annotation makes this a Helm test hook
    "helm.sh/hook": test
    # Delete the pod after the test succeeds (keeps cluster clean)
    "helm.sh/hook-delete-policy": hook-succeeded
spec:
  # restartPolicy must be Never for test pods
  restartPolicy: Never
  containers:
    - name: wget
      image: busybox:1.36
      command:
        - /bin/sh
        - -c
        - |
          # Test that the Service responds on port {{ .Values.service.port }}
          SERVICE={{ .Release.Name }}-nginx.{{ .Release.Namespace }}.svc.cluster.local
          echo "Testing connection to: ${SERVICE}:{{ .Values.service.port }}"
          wget --timeout=5 -qO- http://${SERVICE}:{{ .Values.service.port }} | head -5
          echo "Connection test PASSED!"
EOF

echo "Test written"

### Test 2: Custom Health Check

In [ ]:
cat > /tmp/testchart/templates/tests/test-health.yaml << 'EOF'
apiVersion: v1
kind: Pod
metadata:
  name: {{ .Release.Name }}-test-health
  annotations:
    "helm.sh/hook": test
    "helm.sh/hook-delete-policy": hook-succeeded
    # Weight: health test runs after connection test
    "helm.sh/hook-weight": "5"
spec:
  restartPolicy: Never
  containers:
    - name: health-check
      image: curlimages/curl:8.4.0
      command:
        - /bin/sh
        - -c
        - |
          BASE=http://{{ .Release.Name }}-nginx.{{ .Release.Namespace }}.svc.cluster.local:{{ .Values.service.port }}
          echo "Checking HTTP 200 from ${BASE}/"
          STATUS=$(curl -s -o /dev/null -w "%{http_code}" ${BASE}/)
          if [ "$STATUS" = "200" ]; then
            echo "PASS: Got HTTP ${STATUS}"
            exit 0
          else
            echo "FAIL: Got HTTP ${STATUS}, expected 200"
            exit 1
          fi
EOF

echo "Health check test written"

---

## Rendering the Tests

In [ ]:
# Render all templates including test pods
helm template my-release /tmp/testchart | grep -E 'kind:|name:|helm.sh/hook'

---

## Running Tests Against a Live Release

```bash
# First install the release
helm install my-release /tmp/testchart \
  --namespace test-demo \
  --create-namespace \
  --wait  # wait for all pods to be ready

# Run the tests
helm test my-release --namespace test-demo

# See test pod logs
kubectl logs my-release-test-connection -n test-demo
kubectl logs my-release-test-health -n test-demo
```

Expected output:
```
NAME: my-release
...
TEST SUITE:     my-release-test-connection
Last Started:   ...
Last Completed: ...
Phase:          Succeeded

TEST SUITE:     my-release-test-health  
Phase:          Succeeded
```

In [ ]:
# Lint the chart to verify tests are structurally correct
helm lint /tmp/testchart

---

## Best Practices for Helm Tests

1. **Test what matters most** — connectivity, key HTTP endpoints, core functionality.
2. **Keep tests fast** — ideally < 30 seconds. Long tests block CI pipelines.
3. **Use `hook-delete-policy: hook-succeeded`** — clean up passing pods automatically.
4. **Use specific images** — avoid `latest` in test pods. Pin `busybox:1.36`, `curlimages/curl:8.4.0`.
5. **Run in CI** — add `helm test` as a step after `helm upgrade --install --wait` in your pipeline.

---

## Exercise

In [ ]:
# TASK 1: Add a test that verifies the Deployment has the expected number
#         of replicas (use kubectl inside the test pod — or just test the service).
#         Name the test: <release-name>-test-replicas
# TODO:


# TASK 2: Modify the test-connection pod to also verify the
#         Content-Type header is 'text/html' using curl -I
# TODO:


# TASK 3: If you have a cluster available:
#   a) Deploy the chart: helm install my-release /tmp/testchart -n test-demo --create-namespace --wait
#   b) Run: helm test my-release -n test-demo
#   c) View test pod logs
#   d) Uninstall: helm uninstall my-release -n test-demo
# TODO (live cluster required):

echo "Exercise: implement the tasks above"

---

## Summary

- Test hooks are Pods/Jobs with `"helm.sh/hook": test` annotation.
- `helm test <release>` runs them and reports pass/fail based on exit code.
- Place tests in `templates/tests/` by convention.
- Use `hook-delete-policy: hook-succeeded` to clean up passing test pods.
- Run `helm test` in CI after `helm upgrade --install --wait` for release validation.

---

## Next Lesson

➡️ [Lesson 5.3 — Best Practices & Linting](lesson-5.3-best-practices.ipynb)